# Basic

In [1]:
%load_ext autoreload
%autoreload all

In [2]:
import polars as pl
import pickle
import numpy as np
import tqdm
import os
import glob
import multiprocessing as mp

import src.graph_tokenizer_gd_tree_dev.config as config
import src.graph_tokenizer_gd_tree_dev.tokenizer as tokenizer
import src.graph_tokenizer_gd_tree_dev.eval as eval

In [ ]:
metric_cols = ["semantic_coverage", "conciseness", "distance_score", "uniqueness_entropy",
               "unk_branche_rate", "uncovered_rate", "tree_complexity", "exact_rate", "unique_rate"]

# perf_df_all: greedy_tree_margin (6 lam values) + the baseline heuristics, straight from disk --
# robust to kernel restarts, and this is the persisted, confirmed-complete output of the
# (category, file_type, k) sweep in 4.eval_metrics.ipynb.
perf_df_base = pl.read_parquet(config.Results().perf_baseline_path).rename({"unk_rate": "unk_branche_rate"}).sort(["category", "file_type", "k"])
perf_df_greedy = pl.read_parquet(config.Results().perf_greedy_tree_path).rename({"unk_rate": "unk_branche_rate"}).sort(["category", "file_type", "k"])

# eigenvector_centrality is a degenerate baseline that always returns the same candidate set
# for all k, so it doesn't make sense to include it in the plots.
perf_df_all = pl.concat([perf_df_base, perf_df_greedy], how="diagonal_relaxed").sort(["category", "file_type", "k"]).filter(pl.col("file_type") != "eigenvector_centrality")

In [ ]:
import matplotlib.pyplot as plt

def plot_candidate_performance(perf_df_all, metric_cols):
    """
    2x5 grid of metric-vs-k curves, one line per (category, file_type) series -- last slot
    holds the shared legend. Hue-varied ordinal ramp (red -> yellow -> green) keyed on sorted
    series order; baseline = dotted, greedy_tree_margin = solid, a second encoding on top of
    color so the two arms are distinguishable even without reading the legend.
    """
    series_keys = perf_df_all.select(["category", "file_type"]).unique().sort(["category", "file_type"]).rows()

    cmap = plt.get_cmap("RdYlGn")
    colors = {key: cmap(i / max(len(series_keys) - 1, 1)) for i, key in enumerate(series_keys)}
    linestyles = {key: (":" if key[0] == "baseline" else "-") for key in series_keys}

    fig, axes = plt.subplots(2, 5, figsize=(25, 9))
    axes = axes.flatten()

    for ax, metric in zip(axes, metric_cols):
        for category, file_type in series_keys:
            sub = perf_df_all.filter((pl.col("category") == category) & (pl.col("file_type") == file_type))
            ax.plot(sub["k"], sub[metric], color=colors[(category, file_type)], linestyle=linestyles[(category, file_type)],
                     linewidth=2, marker="o", markersize=3)
        ax.set_title(metric.replace("_", " "))
        ax.set_xlabel("k")
        ax.grid(alpha=0.3)

    axes[-1].axis("off")  # 10th slot unused by metrics -- holds the shared legend instead
    handles = [
        plt.Line2D([0], [0], color=colors[key], linestyle=linestyles[key], lw=2, marker="o", markersize=4,
                   label=f"{key[0]} / {key[1]}")
        for key in series_keys
    ]
    axes[-1].legend(handles=handles, loc="center", fontsize=9, title="category / file_type", frameon=False)

    fig.suptitle("Candidate set performance vs k", fontsize=14)
    fig.tight_layout()
    plt.show()
    return fig


plot_candidate_performance(perf_df_all, metric_cols)

In [ ]:
import src.graph_tokenizer_gd_tree_dev.utils as utils
import networkx as nx
import src.graph_tokenizer_gd_tree_dev.graph_fct as graph_fct


gd_tree_list = glob.glob(config.CandidateLists().path_greedy_tree + '/*.parquet')
baseline_list = glob.glob(config.CandidateLists().baseline_path + '/*.parquet')

all_candidates = {
    "greedy_tree_margin": utils.to_type_dict(gd_tree_list),
    "baseline": utils.to_type_dict(baseline_list),
}
for category, file_list in all_candidates.items():
    for file_type, file in file_list.items():
        print(f"Category: {category}, File Type: {file_type}, File: {file}")

id_to_label, combined_subgraphs = graph_fct.get_combined_combined_subgraphs_and_id2label()
df_mapped = pl.read_parquet(f"{config.BasicConfig().mapped_path}")
mapped_ids = df_mapped["id"].unique().to_list()

gd_tree_list = glob.glob(config.CandidateLists().path_greedy_tree + '/*.parquet')
baseline_list = glob.glob(config.CandidateLists().baseline_path + '/*.parquet')

Ks = config.TokenizerParam().Ks

D = config.TokenizerParam().max_dist_candidate

results = []
candidate_col = "token"
tasks = []
for category, file_list in all_candidates.items():
    if category != "greedy_tree_margin":
        continue
    for file_type, file in file_list.items():
        df = pl.read_parquet(file)
        for k in Ks:
            candidates = df.head(k)[candidate_col].to_list()
            tasks.append(((category, file_type, k), candidates))
tasks

In [ ]:

# Built once and shared across every task instead of being rebuilt per (category, file_type, k):
# A/node_to_idx is the semantic-coverage transition matrix, adj is the out-adjacency used by
# context-tree building. Neither depends on the candidate set T, only on the fixed graph.
A, node_to_idx = tokenizer.build_coverage_transition(combined_subgraphs)
adj = tokenizer.build_out_adjacency(combined_subgraphs)

n_workers = os.cpu_count()
with mp.Pool(n_workers, initializer=tokenizer._init_coverage_worker,
             initargs=(A, node_to_idx, adj, mapped_ids, D, id_to_label)) as pool:
    task_results = pool.imap_unordered(tokenizer._worker_coverage_score, tasks, chunksize=4)
    for (category, file_type, k), metrics in tqdm.tqdm(task_results, total=len(tasks)):
        results.append({
            "category": category,
            "file_type": file_type,
            "k": k,
            **metrics,
        })

results_df = pl.DataFrame(results)
results_df.write_parquet(config.Results().perf_greedy_tree_path)